# 도착예정시간(ETA) 예측

[모델의 목적] 선박이 특정항구에 도착할 예정시간 예측  

[feature]
* 'length', 'beam', 'shiptype', 'mmsi', 'dist1', 'eta(unixtime)', 'draft', 'hdg', 'sog', 'cog', 'rot', 'lat', 'lon', 'lat1', 'lon1', 'latlon1', 'navi', 'utctime', 'ata', 'distance(km)', 'ataport'
* 실제 학습 데이터: length  beam	shiptype	mmsi	eta(unixtime)	draft	hdg	sog	cog	rot	lat1	lon1	utctime	distance(km)	ataport	timedif(ata-utc)

[예측 대상] 'timedif(ata-utc)'(목적지에 도착할 시간과 AIS 메시지 수신한 시간과의 초단위 시간 차이)

[데이터 소스] 해운사가 설치한 안테나와 제휴사로부터 실시간 수집하는 AIS데이터(목적지 예측에 활용된 데이터와 동일하나 정박중, 속도가 비현실적인 경우 등을 제외, 166,321 건  

[알고리즘]  Deep Learning

[정확도] MAE 8600 초

In [ ]:
import pandas as pd
import numpy as np
import random
import tensorflow as tf

print(tf.__version__)

* 로우 데이터(sheet1, sheet2, sheet3)에서 mmsi(선박)가 어느 정도(100개) 이상인 데이터를 각각 추출한다.

In [ ]:
aisdf1 = pd.read_csv('20210823_1_ratio.csv')
aisdf2 = pd.read_csv('20210823_2_ratio.csv')
aisdf3 = pd.read_csv('20210823_3_ratio.csv')

print("df1:",len(aisdf1))
print("df2:",len(aisdf2))
print("df3:",len(aisdf3))

print("\n================================\n")

raw_dataset = pd.concat([aisdf1, aisdf2, aisdf3])
print(len(raw_dataset))
print(raw_dataset.columns)
print()

In [ ]:
aisdf = raw_dataset.drop(['lat','lon','latlon1','ata','dist1','navi'],axis=1)
print(aisdf.head())
len(aisdf['timedif(ata-utc)'].unique())

In [ ]:
aisdf.isnull().sum()

In [ ]:
aisdf = aisdf.dropna()
print(aisdf.isnull().sum())
print('========================\n',len(aisdf['mmsi']))
aisdf.head()

In [ ]:
aisdf_mmsi = aisdf.groupby('mmsi').size()
print(aisdf_mmsi)
print("선박데이터가",len(aisdf_mmsi),'\n=========================')

aisdf_mmsi = aisdf['mmsi'].unique()
print(aisdf_mmsi,'\n==========================')

aisdf_mmsi = aisdf.sort_values(by='mmsi', ascending=True)
print(aisdf_mmsi)


In [ ]:
cntmmsi = aisdf['mmsi'].value_counts(ascending=True)
print(cntmmsi,"====================\n")

In [ ]:
print(cntmmsi.head)

In [ ]:
aisdf.info()

In [ ]:
aisdf.describe()

In [ ]:
timedif = aisdf['timedif(ata-utc)']

y_data = timedif

In [ ]:
aisdf = aisdf.drop('timedif(ata-utc)', axis=1)

X_data = aisdf

In [ ]:
# 피처 스케일링
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_data_scaled = scaler.fit_transform(X_data)

X_data_scaled[0]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_data_scaled, y_data, test_size=0.2, shuffle=True, random_state = 777)

In [ ]:
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

def build_model(num_input=1):
    model = Sequential()
    model.add(Dense(128, activation='relu', input_dim=num_input))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='linear'))
    
#     model.compile(optimizer='adam', loss='mse', metrics=['mae'])
#     model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])    
    opt = keras.optimizers.Adam(learning_rate=0.001)
    model.compile(loss='mse',optimizer=opt,metrics =['mae'])
    return model

In [ ]:
print(len(aisdf.columns))
model = build_model(num_input=15)

In [ ]:
model.fit(X_train, y_train, epochs=2500, batch_size=16, verbose=2)
# model.fit(X_train, y_train, epochs=2500, batch_size=32, verbose=2)

In [ ]:
from sklearn.metrics import mean_absolute_error

predict = model.predict(X_test)

mae = mean_absolute_error(y_test, predict)
print("mae:", mae)

# 2021.10., 2,500 epochs, 7768.783307613187
# 2021.11., 2,500 epochs, 8566.508456786943

In [ ]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, predict)
print("mse:", mse)

In [ ]:
print(len(X_test))

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
# print('y:', y, ',predict:', model.predict(X_test).flatten())

In [ ]:
testdata = X_test[201:300,:]
testpred = model.predict(testdata)
testreal = y_test[201:300]
print(testpred, testreal)

In [ ]:
# 교차 검증
val_model = build_model(num_input=15)
history = val_model.fit(X_train, y_train, batch_size=32, epochs=500, validation_split=0.25, verbose=2)

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curve(total_epoch=10, start=1):
    plt.figure(figsize=(15, 5))
    plt.plot(range(start, total_epoch +1), history.history['loss'][start-1:total_epoch], label='Train')
    plt.plot(range(start, total_epoch +1), history.history['val_loss'][start-1:total_epoch], label='Validation')
    plt.xlabel('Epochs')
    plt.ylabel('mse')
    plt.legend()
    plt.show()

In [ ]:
plot_loss_curve(total_epoch=500, start=1)

In [ ]:
plot_loss_curve(total_epoch=500, start=20)

In [ ]:
from keras.models import load_model
# model.save('eta.h5')
tf.keras.models.save_model('eta.h5')

In [ ]:
from keras.models import load_model
loaded_model = tf.keras.models.load_model('eta.h5')

In [ ]:
testdata = X_test[101:200,:]
testpred = loaded_model.predict(testdata)
testreal = y_test[101:200]
print(testpred, testreal)

In [ ]:
from sklearn.metrics import mean_absolute_error

predict = loaded_model.predict(X_test)

mae = mean_absolute_error(y_test, predict)
print("mae:", mae)